In [ ]:
import pathlib as Path
import numpy as np
import pandas as pd


## Create dataframe from embeddings

In [ ]:
#-----Frequency Embeddings dataframe creation-----
def create_embeddings_dataframe(root_path):
    emb_root = Path.Path(root_path)
    emb_files = emb_root.rglob("embeddings_samples.npz")
    
    all_data = []
    
    for emb_file in emb_files:
        emb = np.load(emb_file)
        X = emb['X']
        y = emb['y']
        subjects = emb['subs']
        
        for i in range(X.shape[0]):
            data_point = {
                'embedding': X[i],
                'label': y[i],
                'subject': subjects[i],
                'file_path': str(emb_file)
            }
            all_data.append(data_point)
    
    df = pd.DataFrame(all_data)
    return df

## Clean embedding DF

In [ ]:
def df_cleaning(df, emb_size=384):
        # --- Expand embedding column into 381 separate columns ---
    embedding_df = pd.DataFrame(df["embedding"].tolist(),
                                columns=[f"emb_{i}" for i in range(emb_size)])

    # --- Concatenate back to the original DataFrame (optional) ---
    df_expanded = pd.concat([df.drop(columns=["embedding"]), embedding_df], axis=1)
    return df_expanded

## Frequency/ Temporal/ Combined DF  

In [ ]:
#-----Frequency Embeddings dataframe creation-----
freq_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//frequency_emb_stored//")
#-----Temporal Embeddings dataframe creation-----
temp_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//temporal_emb_stored//")
#-----Combined Embeddings dataframe creation-----
comb_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//emb_stored//")

freq_emb_df= df_cleaning(freq_emb, emb_size=384)
temp_emb_df= df_cleaning(temp_emb, emb_size=384)
comb_emb_df= df_cleaning(comb_emb, emb_size=768)

# Metadata data labeling
    - 0 -> HC
    - 1 -> unknown
    - 2 -> MCI-AD
    - 3 -> MCI- LBD

In [ ]:
#metadata_path = Path("C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_metadata//preDLB_shared(PSY_RAW)")

metadata_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_metadata//preDLB_shared(PSY_RAW).csv')
df_metadata = pd.read_csv(metadata_path, sep=";", encoding="utf-8-sig")

In [ ]:
clinical_data_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_metadata//clinical_data_csv.csv')
df_clinical = pd.read_csv(clinical_data_path, sep=",", encoding="utf-8-sig")

In [ ]:

import os, json
import numpy as np
import pandas as pd
import optuna, xgboost as xgb
from typing import Dict, Tuple
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
    matthews_corrcoef
)

# def x_y_split(df): ...  # your splitter (must return X, y)
def _safe_feature_names(X, fallback_dim=None):
    if hasattr(X, "columns"):
        return list(X.columns)
    if hasattr(X, "feature_names_in_"):
        return list(X.feature_names_in_)
    if fallback_dim is None and hasattr(X, "shape"):
        fallback_dim = X.shape[1]
    return [f"f{i}" for i in range(int(fallback_dim or 0))]

def _plot_feature_importance(model, feature_names, out_dir: str):
    """Save multiple feature-importance views from XGBoost."""
    import matplotlib.pyplot as plt
    import numpy as np
    from collections import defaultdict
    os.makedirs(out_dir, exist_ok=True)

    # 1) sklearn API importances (gain-based)
    try:
        importances = getattr(model, "feature_importances_", None)
        if importances is not None and len(importances) == len(feature_names):
            order = np.argsort(importances)[::-1]
            top_idx = order[:50]  # limit to top-50 for readability
            plt.figure()
            plt.barh([feature_names[i] for i in top_idx][::-1], importances[top_idx][::-1])
            plt.title("XGB feature_importances_ (top-50)")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "feature_importances_sklearn.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

    # 2) Booster importances (weight/gain/cover)
    try:
        booster = model.get_booster()
        for typ in ["weight", "gain", "cover", "total_gain", "total_cover"]:
            score_dict = booster.get_score(importance_type=typ)
            if not score_dict:
                continue
            # Map 'f0'.. to friendly names
            vals = []
            names = []
            for k, v in score_dict.items():
                if k.startswith("f"):
                    idx = int(k[1:])
                    if 0 <= idx < len(feature_names):
                        names.append(feature_names[idx])
                    else:
                        names.append(k)
                else:
                    names.append(k)
                vals.append(v)
            order = np.argsort(vals)[::-1]
            top = min(50, len(order))
            plt.figure()
            plt.barh([names[i] for i in order[:top]][::-1], np.array(vals)[order[:top]][::-1])
            plt.title(f"Booster feature importance — {typ} (top-{top})")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"feature_importances_{typ}.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

def _save_shap_summaries(model, X_ref, feature_names, out_dir: str, max_display: int = 30, sample: int = 1000):
    """
    Compute SHAP (TreeExplainer) and save bar + beeswarm plots.
    Skips silently if shap not installed or backend issues arise.
    """
    try:
        import shap
        import numpy as np
        import matplotlib.pyplot as plt
    except Exception:
        return

    try:
        # Downsample for speed
        if hasattr(X_ref, "iloc"):
            X_use = X_ref.sample(min(sample, len(X_ref)), random_state=0)
            X_np = X_use.values
        else:
            X_np = X_ref
            if X_np.shape[0] > sample:
                rng = np.random.default_rng(0)
                idx = rng.choice(X_np.shape[0], size=sample, replace=False)
                X_np = X_np[idx]

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_np)

        # Handle binary classification: shap returns array (n, p)
        if isinstance(shap_values, list) and len(shap_values) == 2:
            # pick positive class
            sv = shap_values[1]
        else:
            sv = shap_values

        # Bar plot (mean |SHAP|)
        try:
            plt.figure()
            shap.summary_plot(sv, X_np, feature_names=feature_names, plot_type="bar", show=False, max_display=max_display)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "shap_summary_bar.png"), dpi=200, bbox_inches="tight")
            plt.close()
        except Exception:
            pass

        # Beeswarm
        try:
            plt.figure()
            shap.summary_plot(sv, X_np, feature_names=feature_names, show=False, max_display=max_display)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "shap_summary_beeswarm.png"), dpi=200, bbox_inches="tight")
            plt.close()
        except Exception:
            pass
    except Exception:
        # Any SHAP error -> skip silently
        return

def _pick_device():
    try:
        _ = xgb.core.get_cuda_compute_capabilities()
        return {"tree_method": "hist", "device": "cuda"}
    except Exception:
        return {"tree_method": "hist", "device": "cpu"}

def _ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def _save_json(d: dict, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(d, f, indent=2, ensure_ascii=False)

def _build_cv(n_splits: int, random_state: int, use_loo: bool = False):
    if use_loo:
        return LeaveOneOut()
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

def make_xgb_objective(X, y, scoring: str = "balanced_accuracy", n_splits: int = 5, random_state: int = 42, use_loo: bool = False):
    if scoring == "balanced_accuracy":
        from sklearn.metrics import balanced_accuracy_score as metric_fn
        needs_proba = False
        eval_metric = "logloss"
    elif scoring == "roc_auc":
        from sklearn.metrics import roc_auc_score as metric_fn
        needs_proba = True
        eval_metric = "auc"
    else:
        raise ValueError("scoring must be 'balanced_accuracy' or 'roc_auc'")

    folds = _build_cv(n_splits=n_splits, random_state=random_state, use_loo=use_loo)
    device_kwargs = _pick_device()

    pos_ratio = float(np.mean(y))
    scale_pos_weight = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0

    def objective(trial: optuna.Trial) -> float:
        params = {
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 200, 1500),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "objective": "binary:logistic",
            "eval_metric": eval_metric,
            "n_jobs": 1,  # keep each trial single-threaded
            "scale_pos_weight": scale_pos_weight,
            **device_kwargs,
        }

        scores = []
        for tr_idx, va_idx in folds.split(X, y):
            if hasattr(X, "iloc"):
                X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
                y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            else:
                X_tr, X_va = X[tr_idx], X[va_idx]
                y_tr, y_va = y[tr_idx], y[va_idx]

            model = xgb.XGBClassifier(**params)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                #early_stopping_rounds=50,
                verbose=False,
            )

            if needs_proba:
                y_score = model.predict_proba(X_va)[:, 1]
                score = metric_fn(y_va, y_score)
            else:
                y_pred = model.predict(X_va)
                score = metric_fn(y_va, y_pred)
            scores.append(score)

            trial.report(np.mean(scores), len(scores))
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(scores))

    return objective

def _evaluate_and_save(name: str, model, X_test, y_test, out_dir: str, feature_names=None, X_ref_for_shap=None):
    """Compute metrics, plot ROC + confusion matrix, and save everything."""
    _ensure_dir(out_dir)

    # Predictions
    y_proba = None
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
    except Exception:
        pass
    y_pred = model.predict(X_test)

    # Metrics
    metrics = {
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "mcc": None,
        "roc_auc": None,
        "sensitivity": None,   # <-- NEW
        "specificity": None,   # <-- NEW
        "classification_report": None,
    }
    # MCC
    try:
        metrics["mcc"] = float(matthews_corrcoef(y_test, y_pred))
    except Exception:
        pass
    # ROC-AUC if both classes present and proba available
    try:
        if y_proba is not None and len(np.unique(y_test)) == 2:
            metrics["roc_auc"] = float(roc_auc_score(y_test, y_proba))
        else:
            metrics["roc_auc"] = None
    except Exception:
        metrics["roc_auc"] = None
    # --- Sensitivity & Specificity (labels fixed to [0,1]) ---
    try:
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        sens_den = tp + fn
        spec_den = tn + fp
        metrics["sensitivity"] = float(tp / sens_den) if sens_den > 0 else None
        metrics["specificity"] = float(tn / spec_den) if spec_den > 0 else None
    except Exception:
        pass


    # Classification report (as dict)
    try:
        metrics["classification_report"] = classification_report(y_test, y_pred, output_dict=True)
    except Exception:
        metrics["classification_report"] = None

    # Confusion matrix plot
    try:
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(cm)
        import matplotlib.pyplot as plt
        plt.figure()
        disp.plot(values_format="d")
        plt.title(f"Confusion Matrix — {name}")
        plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=200, bbox_inches="tight")
        plt.close()
    except Exception:
        pass

    # ROC curve plot
    try:
        if y_proba is not None and len(np.unique(y_test)) == 2:
            import matplotlib.pyplot as plt
            plt.figure()
            RocCurveDisplay.from_predictions(y_test, y_proba)
            plt.title(f"ROC Curve — {name}")
            plt.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

    # Feature names
    if feature_names is None:
        feature_names = _safe_feature_names(X_test)

    # Feature importance plots
    try:
        _plot_feature_importance(model, feature_names, out_dir)
    except Exception:
        pass

    # SHAP plots (optional)
    try:
        if X_ref_for_shap is None:
            X_ref_for_shap = X_test
        _save_shap_summaries(model, X_ref_for_shap, feature_names, out_dir)
    except Exception:
        pass

    # Save metrics JSON
    _save_json(metrics, os.path.join(out_dir, "metrics.json"))
    return metrics

def _evaluate_cv_and_save(
    name: str,
    X,
    y,
    params: dict,
    scoring: str,
    n_splits: int,
    random_state: int,
    out_dir: str,
    use_loo: bool = False,
):
    """Evaluate metrics using CV on the full dataset and save plots/metrics."""
    _ensure_dir(out_dir)

    folds = _build_cv(n_splits=n_splits, random_state=random_state, use_loo=use_loo)

    y_true_all = []
    y_pred_all = []
    y_proba_all = []
    has_proba = True
    fold_metrics = []

    for tr_idx, va_idx in folds.split(X, y):
        if hasattr(X, "iloc"):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        else:
            X_tr, X_va = X[tr_idx], X[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            #early_stopping_rounds=50,
            verbose=False,
        )

        y_pred = model.predict(X_va)
        y_proba = None
        if has_proba:
            try:
                y_proba = model.predict_proba(X_va)[:, 1]
            except Exception:
                has_proba = False
                y_proba = None

        # Fold-level metrics
        fold_result = {
            "balanced_accuracy": float(balanced_accuracy_score(y_va, y_pred)),
            "mcc": None,
            "roc_auc": None,
            "sensitivity": None,
            "specificity": None,
        }
        try:
            fold_result["mcc"] = float(matthews_corrcoef(y_va, y_pred))
        except Exception:
            pass
        try:
            if y_proba is not None and len(np.unique(y_va)) == 2:
                fold_result["roc_auc"] = float(roc_auc_score(y_va, y_proba))
        except Exception:
            pass
        try:
            cm = confusion_matrix(y_va, y_pred, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()
            sens_den = tp + fn
            spec_den = tn + fp
            fold_result["sensitivity"] = float(tp / sens_den) if sens_den > 0 else None
            fold_result["specificity"] = float(tn / spec_den) if spec_den > 0 else None
        except Exception:
            pass
        fold_metrics.append(fold_result)

        y_true_all.append(y_va)
        y_pred_all.append(y_pred)
        if has_proba:
            y_proba_all.append(y_proba)

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)
    y_proba_all = np.concatenate(y_proba_all) if has_proba and len(y_proba_all) else None

    # Pooled metrics
    metrics = {
        "balanced_accuracy": float(balanced_accuracy_score(y_true_all, y_pred_all)),
        "mcc": None,
        "roc_auc": None,
        "sensitivity": None,
        "specificity": None,
        "classification_report": None,
        "cv_stats": {},
    }
    try:
        metrics["mcc"] = float(matthews_corrcoef(y_true_all, y_pred_all))
    except Exception:
        pass
    try:
        if y_proba_all is not None and len(np.unique(y_true_all)) == 2:
            metrics["roc_auc"] = float(roc_auc_score(y_true_all, y_proba_all))
    except Exception:
        pass
    try:
        cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        sens_den = tp + fn
        spec_den = tn + fp
        metrics["sensitivity"] = float(tp / sens_den) if sens_den > 0 else None
        metrics["specificity"] = float(tn / spec_den) if spec_den > 0 else None
    except Exception:
        pass
    try:
        metrics["classification_report"] = classification_report(y_true_all, y_pred_all, output_dict=True)
    except Exception:
        metrics["classification_report"] = None

    # Fold summary stats
    for key in ["balanced_accuracy", "mcc", "roc_auc", "sensitivity", "specificity"]:
        vals = [m.get(key) for m in fold_metrics if m.get(key) is not None]
        metrics["cv_stats"][f"{key}_mean"] = float(np.mean(vals)) if vals else None
        metrics["cv_stats"][f"{key}_std"] = float(np.std(vals)) if vals else None

    # Confusion matrix plot
    try:
        cm = confusion_matrix(y_true_all, y_pred_all)
        disp = ConfusionMatrixDisplay(cm)
        import matplotlib.pyplot as plt
        plt.figure()
        disp.plot(values_format="d")
        plt.title(f"Confusion Matrix ? {name}")
        plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=200, bbox_inches="tight")
        plt.close()
    except Exception:
        pass

    # ROC curve plot
    try:
        if y_proba_all is not None and len(np.unique(y_true_all)) == 2:
            import matplotlib.pyplot as plt
            plt.figure()
            RocCurveDisplay.from_predictions(y_true_all, y_proba_all)
            plt.title(f"ROC Curve ? {name}")
            plt.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

    _save_json(metrics, os.path.join(out_dir, "metrics.json"))
    return metrics

def optimize_many(
    dfs: Dict[str, pd.DataFrame],
    x_y_split_fn,
    scoring: str = "balanced_accuracy",
    n_trials: int = 500,
    n_splits: int = 5,
    random_state: int = 42,
    n_jobs_trials: int = -1,
    output_dir: str = "xgb_results",
    test_size: float = 0.20,
    split_param: str = "emb_",
    use_loo: bool = False,
) -> Tuple[pd.DataFrame, Dict[str, xgb.XGBClassifier], Dict[str, optuna.Study]]:
    """
    Runs Optuna per dataset on the full dataset and validates via CV,
    saving plots + metrics in output_dir/<dataset_name>/
    """
    results = []
    best_models: Dict[str, xgb.XGBClassifier] = {}
    studies: Dict[str, optuna.Study] = {}

    _ensure_dir(output_dir)

    for name, df in dfs.items():
        X, y = x_y_split_fn(df, split_param=split_param)

        # To numpy for fast indexing, but keep DataFrame support
        X_arr = X.values if hasattr(X, "values") else X
        y_arr = y.values if hasattr(y, "values") else y

        feature_names = _safe_feature_names(X)
        # Keep names before converting to numpy

        # Build objective on full dataset
        objective = make_xgb_objective(
            X_arr, y_arr,
            scoring=scoring,
            n_splits=n_splits,
            random_state=random_state,
            use_loo=use_loo,
        )

        study = optuna.create_study(
            study_name=f"{name}_{scoring}",
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=random_state),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=10),
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True, n_jobs=n_jobs_trials)
        studies[name] = study

        # Train final model on the full dataset using best params
        device_kwargs = _pick_device()
        pos_ratio = float(np.mean(y_arr))
        spw = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0

        best_params = {
            **study.best_params,
            "objective": "binary:logistic",
            "eval_metric": "auc" if scoring == "roc_auc" else "logloss",
            "n_jobs": 0,
            "scale_pos_weight": spw,
            **device_kwargs,
        }
        final_model = xgb.XGBClassifier(**best_params)
        final_model.fit(X_arr, y_arr, verbose=False)

        # Save per-dataset artifacts
        ds_out = os.path.join(output_dir, name)
        _ensure_dir(ds_out)

        # CV evaluation on full dataset
        metrics = _evaluate_cv_and_save(
            name,
            X_arr,
            y_arr,
            best_params,
            scoring=scoring,
            n_splits=n_splits,
            random_state=random_state,
            out_dir=ds_out,
            use_loo=use_loo,
        )

        # Save best params + study summary
        _save_json({"best_params": study.best_params, "best_value": study.best_value}, os.path.join(ds_out, "best.json"))

        # Feature importance plots
        try:
            _plot_feature_importance(final_model, feature_names, ds_out)
        except Exception:
            pass

        # SHAP plots (optional)
        try:
            _save_shap_summaries(final_model, X_arr, feature_names, ds_out)
        except Exception:
            pass

        # Optional: save model
        try:
            import joblib
            joblib.dump(final_model, os.path.join(ds_out, "model.joblib"))
        except Exception:
            pass

        best_models[name] = final_model
        results.append({
            "dataset": name,
            "cv_best_value": study.best_value,
            "cv_balanced_accuracy": metrics.get("balanced_accuracy"),
            "cv_roc_auc": metrics.get("roc_auc"),
            "n_trials": len(study.trials),
        })

    summary_df = pd.DataFrame(results).sort_values(by="cv_balanced_accuracy", ascending=False).reset_index(drop=True)
    # Save a global summary too
    summary_df.to_csv(os.path.join(output_dir, "summary.csv"), index=False)
    return summary_df, best_models, studies


In [ ]:
handcrafted_features_path = 'C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_hf//corpus_LBD_CZ_002_writing_results_table_original_filtered.csv'
df_handcrafted_features = pd.read_csv(handcrafted_features_path, sep=";", encoding="utf-8-sig")

In [ ]:
def task_wise_df_creation(df, task_name):
    task_df = df[df['file_path'].str.contains(task_name)].reset_index(drop=True)
    return task_df

In [ ]:
import re
import pandas as pd

def make_task_embedding_wide(
    df: pd.DataFrame,
    id_cols=("subject", "diagnosis"),      # adjust if you don't have 'diagnosis'
    emb_prefix="emb_",
    task_regex=r"(\d+_\d+)"                 # extracts e.g. "1_1" from file_path
):
    df = df.copy()

    # --- 1) Ensure we have an explicit 'task' column (parse from file_path)
    if "task" not in df.columns:
        def extract_task(fp):
            m = re.search(task_regex, str(fp))
            return m.group(1) if m else None
        df["task"] = df["file_path"].apply(extract_task)

    # --- 2) Identify embedding columns
    emb_cols = [c for c in df.columns if c.startswith(emb_prefix)]

    # --- 3) If multiple rows per (id_cols, task), aggregate (mean) to one row
    group_cols = [c for c in id_cols if c in df.columns] + ["task"]
    agg_df = (
        df[group_cols + emb_cols]
        .groupby(group_cols, dropna=False, as_index=False)
        .mean(numeric_only=True)
    )

    # --- 4) Pivot to wide: columns = (task, emb_k) → emb_k_<task>
    long_df = agg_df.melt(id_vars=group_cols, value_vars=emb_cols,
                          var_name="emb", value_name="value")
    wide = long_df.pivot_table(
        index=group_cols[:-1],    # all id cols except 'task'
        columns=["task", "emb"],
        values="value"
    )

    # --- 5) Flatten MultiIndex columns into 'emb_k_<task>'
    wide.columns = [f"{emb}_{task}" for task, emb in wide.columns]
    wide = wide.reset_index()

    # Optional: sort columns (keep id cols first)
    id_keep = [c for c in id_cols if c in wide.columns]
    other_cols = sorted([c for c in wide.columns if c not in id_keep])
    wide = wide[id_keep + other_cols]

    return wide

# ---------------------
# Example usage:
# wide_df = make_task_embedding_wide(df)
# print(wide_df.shape)
# print(wide_df.columns[:10])


In [ ]:
import pandas as pd

def add_suffix_to_emb_cols(df: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """
    Adds a given string suffix to all columns containing 'emb'.

    Example:
        add_suffix_to_emb_cols(df, '_1_1')
        -> emb_0 becomes emb_0_1_1, emb_1 becomes emb_1_1_1, etc.
    """
    df = df.copy()
    rename_dict = {
        col: f"{col}{suffix}" for col in df.columns if "emb" in col
    }
    return df.rename(columns=rename_dict)


In [ ]:
freq_emb_df_wide = make_task_embedding_wide(freq_emb_df)

In [ ]:
freq_emb_df_wide = add_suffix_to_emb_cols(freq_emb_df_wide, '_freq')

In [ ]:

temp_emb_df_wide = make_task_embedding_wide(temp_emb_df)

In [ ]:
temp_emb_df_wide = add_suffix_to_emb_cols(temp_emb_df_wide, '_temp')

In [ ]:
#df_emb_wide = pd.concat([mfreq_emb_df_wide, temp_emb_df_wide.drop(columns=['subject'])], axis=1)
df_emb_wide = pd.concat([freq_emb_df_wide, temp_emb_df_wide], axis=1)
df_emb_wide = df_emb_wide.loc[:,~df_emb_wide.columns.duplicated()].copy()

In [ ]:
import pandas as pd

def merge_on_subject(df1, df2):
    """
    Merge two DataFrames side-by-side for matching 'subject' values only.
    Keeps only subjects that exist in both DataFrames.
    Automatically renames 'ID' to 'subject' if necessary.
    """
    # Create copies to avoid modifying the originals
    df1 = df1.copy()
    df2 = df2.copy()

    # Ensure both have a 'subject' column
#    if 'subject' not in df1.columns:
#        df1 = df1.rename(columns={'ID': 'subject'})
    if 'subject' not in df2.columns:
        df2 = df2.rename(columns={'ID': 'subject'})

    # Perform a one-to-one merge on 'subject'
    merged_df = pd.merge(
        df1,
        df2,
        on='subject',
        how='inner',
        validate='one_to_one'
    )
    return merged_df



In [ ]:
df_full_features = merge_on_subject(df_emb_wide, df_handcrafted_features)

In [ ]:
import pandas as pd

def append_column_by_partial_match(df_main, df_source, match_col_main, match_col_source, value_col, new_col_name):
    """
    Append a column to df_main from df_source where df_main[match_col_main]
    partially matches df_source[match_col_source].
    Only non-missing values from df_source[value_col] are appended.

    Parameters
    ----------
    df_main : pd.DataFrame
        Target DataFrame to append the column to.
    df_source : pd.DataFrame
        Source DataFrame containing the column to append.
    match_col_main : str
        Column in df_main used for matching (substring search).
    match_col_source : str
        Column in df_source to match against.
    value_col : str
        Column in df_source containing values to append.
    new_col_name : str
        Name of the new column to create in df_main.
    """
    df_main = df_main.copy()
    df_source = df_source.copy()

    # Drop missing values from the source value column
    df_source = df_source.dropna(subset=[value_col])

    # Initialize new column with NaN
    df_main[new_col_name] = pd.NA

    # Perform partial string match row by row
    for i, subj in enumerate(df_source[match_col_source].astype(str)):
        matches = df_source[df_source[match_col_source].astype(str).str.contains(subj, na=False, case=False)]
        if not matches.empty:
            # Take the first match or handle multiple matches as desired
            df_main.at[i, new_col_name] = matches[value_col].iloc[0]

    return df_main


In [ ]:
import re
import pandas as pd

def append_col_when_main_contains_source(
    df_main, df_source, *, 
    match_col_main="subject",            # in df_main
    match_col_source="ID_1.meranie",     # in df_source
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
):
    df_main = df_main.copy()
    df_source = df_source.copy()

    # normalize + keep only rows in source with non-missing values
    df_main[match_col_main] = df_main[match_col_main].astype(str).str.strip()
    df_source[match_col_source] = df_source[match_col_source].astype(str).str.strip()
    df_source = df_source.dropna(subset=[value_col])

    # init target column
    if new_col_name not in df_main:
        df_main[new_col_name] = pd.NA

    # for each source row, mark all main rows whose subject CONTAINS the source token
    for _, r in df_source.iterrows():
        token = r[match_col_source]
        if not token:
            continue
        mask = df_main[match_col_main].str.contains(re.escape(token), na=False, case=not case)
        # write only where we don't have a value yet (keeps first hit)
        to_set = mask & df_main[new_col_name].isna()
        df_main.loc[to_set, new_col_name] = r[value_col]

    return df_main


In [ ]:
# ...existing code...
def append_multiple_cols_when_main_contains_source(
    df_main, df_source, *,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_cols=("HC0_nHC1_MCI2_MCILB3_baseline",),
    case=False
):
    df_main = df_main.copy()
    df_source = df_source.copy()

    # normalize + clean
    df_main[match_col_main] = df_main[match_col_main].astype(str).str.strip()
    df_source[match_col_source] = df_source[match_col_source].astype(str).str.strip()

    # initialize missing columns
    for col in value_cols:
        if col not in df_main.columns:
            df_main[col] = pd.NA

    # iterate through source
    for _, r in df_source.iterrows():
        token = r[match_col_source]
        if not token or pd.isna(token):
            continue

        # use 'case' argument as passed (was inverted before)
        mask = df_main[match_col_main].str.contains(re.escape(str(token)), na=False, case=case)
        to_set = mask

        # assign all defined value columns
        for col in value_cols:
            if col not in r or pd.isna(r[col]):
                continue
            df_main.loc[to_set & df_main[col].isna(), col] = r[col]

    return df_main
# ...existing code...

In [ ]:
df_full_features_lbl = append_col_when_main_contains_source(
    df_main=df_full_features,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)


In [ ]:
df_full_features_lbl.to_csv("LBD_CZ_002_dfs/df_full_features_lbl.csv", index=False)

In [ ]:

freq_emb_df_lbl = append_col_when_main_contains_source(
    df_main=freq_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [ ]:
freq_emb_df_lbl.to_csv("LBD_CZ_002_dfs/freq_emb_df_lbl.csv", index=False)

In [ ]:
temp_emb_df_lbl = append_col_when_main_contains_source(
    df_main=temp_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [ ]:
temp_emb_df_lbl.to_csv("LBD_CZ_002_dfs/temp_emb_df_lbl.csv", index=False)

In [ ]:
comb_emb_df_lbl = append_col_when_main_contains_source(
    df_main=comb_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)


In [ ]:
comb_emb_df_lbl.to_csv("LBD_CZ_002_dfs/comb_emb_df_lbl.csv", index=False)

In [ ]:

df_handcrafted_lbl = append_col_when_main_contains_source(
df_handcrafted_lbl = append_col_when_main_contains_source(
    df_main=df_handcrafted_features,
    df_source=df_metadata,
    match_col_main="ID",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

    df_main=df_handcrafted_features,
    df_source=df_metadata,
    match_col_main="ID",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)


In [ ]:
df_handcrafted_lbl.to_csv("LBD_CZ_002_dfs/df_handcrafted_lbl.csv", index=False)

In [ ]:
df_handcrafted_meta = append_multiple_cols_when_main_contains_source(
    df_main=df_handcrafted_features,
    df_source=df_metadata,
    match_col_main="ID",
    match_col_source="ID_1.meranie",
    value_cols=["HC0_nHC1_MCI2_MCILB3_baseline","Delka_vzdelani"],
)

In [ ]:
df_handcrafted_clinical_lbl = append_multiple_cols_when_main_contains_source(
    df_main=df_handcrafted_lbl,
    df_source=df_clinical,
    match_col_main="ID",
    match_col_source="#personalID",
    value_cols=["age",
                "gender",
                "MOCA",
                "education type",
                "education length",
                "memory z-score",
                "visuo-spatial z-score",
                "attention z-score",
                "executive function z-score",
                "GDS"]
)

In [ ]:
df_handcrafted_clinical_lbl = df_handcrafted_clinical_lbl.dropna(subset=['age'], axis=0)

In [ ]:

df_handcrafted_clinical_lbl.to_csv("LBD_CZ_002_dfs/df_handcrafted_clinical_lbl.csv", index=False)

# Spearsman correlation / finding covarriants

In [ ]:
cols_to_convert = df_handcrafted_clinical_lbl.columns[1:]

df_handcrafted_clinical_lbl[cols_to_convert] = (
    df_handcrafted_clinical_lbl[cols_to_convert]
    .apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', '.'), errors='coerce'))
)

In [ ]:
df_handcrafted_clinical_lbl.dtypes

In [ ]:
df_handcrafted_clinical_lbl

In [ ]:
from scipy import stats
import numpy as np

meta_cols=["age",
         #   "gender",
            "MOCA",
            "education type",
            "education length",
            "memory z-score",
            "visuo-spatial z-score",
            "attention z-score",
            "executive function z-score",
            "GDS"]

feature_cols = df_handcrafted_clinical_lbl.drop(columns=['ID', 'diagnosis','gender'] + meta_cols).columns.tolist()


def correlation_to_df(df, feature_cols, meta_cols, corr_type='spearman'):
    from scipy import stats
    corr_results = []
    for feature in feature_cols:
        for meta in meta_cols:
            feature_data = pd.to_numeric(df[feature], errors='coerce')
            meta_data = pd.to_numeric(df[meta], errors='coerce')
            if corr_type == 'spearman':
                corr, p_value = stats.spearmanr(feature_data, meta_data, nan_policy='omit')
            elif corr_type == 'pearson':
                corr, p_value = stats.pearsonr(feature_data.dropna(), meta_data.dropna())
            else:
                raise ValueError("corr_type must be 'spearman' or 'pearson'")
            corr_results.append({
                'feature': feature,
                'meta_variable': meta,
                'correlation': corr,
                'p_value': p_value
            })
    corr_df = pd.DataFrame(corr_results)
    return corr_df


In [ ]:
df_hf_clinical_corr = correlation_to_df(df_handcrafted_clinical_lbl, feature_cols, meta_cols, corr_type='spearman')

In [ ]:
df_hf_clinical_corr

In [ ]:
# Create separate pivots for correlation and p-value
df_corr_pivot = df_hf_clinical_corr.pivot(
    index='feature',
    columns='meta_variable',
    values='correlation'
)

df_pval_pivot = df_hf_clinical_corr.pivot(
    index='feature',
    columns='meta_variable',
    values='p_value'
)

In [ ]:
df_pval_pivot.to_csv("LBD_CZ_002_dfs/df_hf_clinical_corr_pvalues.csv")

In [ ]:
df_pval_pivot[df_pval_pivot < 0.05].count()

## FDR correction

In [ ]:
def p_value_filter(df, p_col="p_value", alpha=0.05):
    from statsmodels.stats.multitest import multipletests
    reject, p_corr, _, _ = multipletests(df[p_col], alpha=alpha, method="fdr_bh")
    out = df.copy()
    out["p_fdr"] = p_corr
    out["significant"] = reject
    return out[out["significant"]]


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

def fdr_correction_matrix(df_pvals: pd.DataFrame, alpha: float = 0.05, method: str = "fdr_bh"):
    df = df_pvals.copy()

    # Only numeric columns (skip "feature")
    numeric_cols = df.select_dtypes(include=[float, int]).columns
    arr = df[numeric_cols].to_numpy(dtype=float)

    # Mask NaNs
    mask_na = np.isnan(arr)

    # Take only valid p-values (flatten)
    pvals_flat = arr[~mask_na]

    # FDR
    reject, pvals_corr, _, _ = multipletests(pvals_flat, alpha=alpha, method=method)

    # Prepare output arrays
    arr_fdr = np.full_like(arr, np.nan, dtype=float)
    arr_sig = np.zeros_like(arr, dtype=bool)

    # Fill corrected values
    arr_fdr[~mask_na] = pvals_corr
    arr_sig[~mask_na] = reject

    # Back to DataFrames
    df_fdr = df.copy()
    df_sig = df.copy()

    df_fdr[numeric_cols] = arr_fdr
    df_sig[numeric_cols] = arr_sig

    return df_fdr, df_sig


In [ ]:
df_fdr, df_sig = fdr_correction_matrix(df_pval_pivot, alpha=0.05)

#Show significant (feature, metadata) pairs
significant_pairs = (
    df_sig
         .iloc[:, 1:]         # skip the 'feature' column
         .stack()             # long format
         .loc[lambda s: s]    # keep only True
         .index
         .to_frame(name=["feature", "metadata"])
)
##
significant_pairs


In [ ]:
df_after_fdr = df_fdr[df_sig]

In [ ]:
df_after_fdr.dropna(how='all', inplace=True)
df_after_fdr

In [ ]:
list_to_compensate = df_after_fdr.index.tolist()

In [ ]:
list_to_compensate

In [ ]:
list_to_compensate.remove("w.cz.fnusa.17_1_slope of velocity (in-air)")

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

def residualize_features_against_gds(X, gds):
    X_res = X.copy()
    gds = gds.fillna(gds.mean())
    lr = LinearRegression()
    for col in X.columns:
        lr.fit(gds.values.reshape(-1, 1), X[col])
        X_res[col] = X[col] - lr.predict(gds.values.reshape(-1, 1))
    return X_res


In [ ]:
df_handcrafted_clinical_lbl_res = df_handcrafted_clinical_lbl.copy()
df_handcrafted_clinical_lbl_res[list_to_compensate] = residualize_features_against_gds(
    df_handcrafted_clinical_lbl[list_to_compensate],
    df_handcrafted_clinical_lbl["GDS"]
)

# Task-wise Dataframe dictionaries creation

In [ ]:
task_list = ['1_1', '9_1', '15_1', '16_1', '17_1', '18_1', '19_1']

task_freq_dfs = {task: task_wise_df_creation(freq_emb_df_lbl, task) for task in task_list}
task_temp_dfs = {task: task_wise_df_creation(temp_emb_df_lbl, task) for task in task_list}
task_comb_dfs = {task: task_wise_df_creation(comb_emb_df_lbl, task) for task in task_list}


In [ ]:
task_list = ['1_1', '9_1', '15_1', '16_1', '17_1', '18_1', '19_1']

task_hf_dfs = {
    task: df_handcrafted_lbl.loc[
        :,
        df_handcrafted_lbl.columns.astype(str).str.contains(task, case=False) |
        df_handcrafted_lbl.columns.isin(['diagnosis', 'ID'])
    ].copy()
    for task in task_list
}

In [ ]:

task_list = ['1_1', '9_1', '15_1', '16_1', '17_1', '18_1', '19_1']

task_compensated_dfs = {
    task: df_handcrafted_clinical_lbl_res.loc[
        :,
        df_handcrafted_clinical_lbl_res.columns.astype(str).str.contains(task, case=False) |
        df_handcrafted_clinical_lbl_res.columns.isin(['diagnosis', 'ID'])
    ].copy()
    for task in task_list
}

### MCI-LBD vs HC

In [ ]:
task_freq_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2])].reset_index(drop=True) for k, v in task_freq_dfs.items()}
task_temp_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2])].reset_index(drop=True) for k, v in task_temp_dfs.items()}
task_comb_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2])].reset_index(drop=True) for k, v in task_comb_dfs.items()}
task_hf_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2])].reset_index(drop=True) for k, v in task_hf_dfs.items()}
task_compensated_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2])].reset_index(drop=True) for k, v in task_compensated_dfs.items()}


### MCI-AD vs HC

In [ ]:
task_freq_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3])].reset_index(drop=True) for k, v in task_freq_dfs.items()}
task_temp_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3])].reset_index(drop=True) for k, v in task_temp_dfs.items()}
task_comb_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3])].reset_index(drop=True) for k, v in task_comb_dfs.items()}
task_hf_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3])].reset_index(drop=True) for k, v in task_hf_dfs.items()}
task_compensated_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3])].reset_index(drop=True) for k, v in task_compensated_dfs.items()}

### MCI-LBD vs MCI-AD

In [ ]:
task_freq_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0])].reset_index(drop=True) for k, v in task_freq_dfs.items()}
task_temp_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0])].reset_index(drop=True) for k, v in task_temp_dfs.items()}
task_comb_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0])].reset_index(drop=True) for k, v in task_comb_dfs.items()}
task_hf_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0])].reset_index(drop=True) for k, v in task_hf_dfs.items()}
task_compensated_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0])].reset_index(drop=True) for k, v in task_compensated_dfs.items()}

In [ ]:
task_freq_dfs_lbd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs_lbd_hc.items()}
task_temp_dfs_lbd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs_lbd_hc.items()}
task_comb_dfs_lbd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs_lbd_hc.items()}

In [ ]:

task_freq_dfs_ad_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs_ad_hc.items()}
task_temp_dfs_ad_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs_ad_hc.items()}
task_comb_dfs_ad_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs_ad_hc.items()}

In [ ]:

task_freq_dfs_ad_lbd = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs_ad_lbd.items()}
task_temp_dfs_ad_lbd = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs_ad_lbd.items()}
task_comb_dfs_ad_lbd = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs_ad_lbd.items()}

In [ ]:
task_freq_dfs_pd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs.items()}
task_temp_dfs_pd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs.items()}
task_comb_dfs_pd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs.items()}

In [ ]:
for dfs in (task_freq_dfs_lbd_hc, task_temp_dfs_lbd_hc, task_comb_dfs_lbd_hc, task_hf_dfs_lbd_hc, task_compensated_dfs_lbd_hc):
    
    for k, v in dfs.items():
        if "diagnosis" in v.columns:
            v["diagnosis"] = v["diagnosis"].replace({3.0: 1, 0.0: 0})


In [ ]:
for dfs in (task_freq_dfs_ad_hc, task_temp_dfs_ad_hc, task_comb_dfs_ad_hc, task_hf_dfs_ad_hc, task_compensated_dfs_ad_hc):
    
    for k, v in dfs.items():
        if "diagnosis" in v.columns:
            v["diagnosis"] = v["diagnosis"].replace({2.0: 1, 0.0: 0})


In [ ]:

for dfs in (task_freq_dfs_ad_lbd, task_temp_dfs_ad_lbd, task_comb_dfs_ad_lbd, task_hf_dfs_ad_lbd, task_compensated_dfs_ad_lbd):
    
    for k, v in dfs.items():
        if "diagnosis" in v.columns:
            v["diagnosis"] = v["diagnosis"].replace({3.0: 1, 2.0: 0})

In [ ]:
def diagnosis_mapping(df, col_name="diagnosis", case = "HC-MLI-LB"):
    # HC-MLI-LB: map 0.0 -> 0 (HC), 3.0 -> 1 (MLI-LB)
    match case:
        case "HC-MLI-LB":
    # HC-MLI-LB: map 0.0 -> 0 (HC), 3.0 -> 1 (MLI-LB)
            df[col_name] = df[col_name].replace({0.0: 0, 3.0: 1})
        case "HC-MLI-AD":
    # HC-MLI-LB: map 0.0 -> 0 (HC), 2.0 -> 1 (MLI-AD)
            df[col_name] = df[col_name].replace({0.0: 0, 2.0: 1})
        case "MLI-LB-MLI-AD":
    # HC-MLI-LB: map 3.0 -> 0 (MLI-LBD), 2.0 -> 1 (MLI-AD)
            df[col_name] = df[col_name].replace({3.0: 0, 2.0: 1})
        case _:
            raise ValueError(f"Unknown mapping case: {case}")

In [ ]:
def miss_val_clean(df, percent_threshold=0.8):
    """
    Cleans the DataFrame by removing columns with more than the specified percentage of missing values.
    
    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame to be cleaned.
    percent_threshold : float
        The maximum allowed percentage of missing values in a column (between 0 and 1).
        
    Returns
    -------
    pd.DataFrame
        The cleaned DataFrame with columns exceeding the missing value threshold removed.
    """
    df_cleaned = df.copy()
    df_cleaned = df_cleaned.loc[:, df_cleaned.isnull().mean() <= percent_threshold]
    
    cols_to_convert = df_cleaned.columns[1:-1]

    df_cleaned[cols_to_convert] = (
        df_cleaned[cols_to_convert]
        .apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', '.'), errors='coerce'))
    )

    df_cleaned = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))
    return df_cleaned

In [ ]:
task_hf_dfs_clean_lbd_hc = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_hf_dfs_lbd_hc.items()}
task_compensated_dfs_clean_lbd_hc = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_compensated_dfs_lbd_hc.items()}
task_hf_dfs_clean_ad_hc = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_hf_dfs_ad_hc.items()}
task_compensated_dfs_clean_ad_hc = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_compensated_dfs_ad_hc.items()}
task_hf_dfs_clean_ad_lbd = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_hf_dfs_ad_lbd.items()}
task_compensated_dfs_clean_ad_lbd = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_compensated_dfs_ad_lbd.items()}

In [ ]:
# Use this one 
def merge_on_subject_agg_right(df1, df2):
    df1 = df1.copy(); df2 = df2.copy()
    if 'subject' not in df1 and 'ID' in df1: df1 = df1.rename(columns={'ID':'subject'})
    if 'subject' not in df2 and 'ID' in df2: df2 = df2.rename(columns={'ID':'subject'})
    num_cols = df2.select_dtypes('number').columns.tolist()
    agg = {c:'first' for c in df2.columns if c not in num_cols and c!='subject'}
    agg.update({c:'mean' for c in num_cols})
    df2 = df2.groupby('subject', as_index=False).agg(agg)
    return pd.merge(df1, df2, on='subject', how='inner', validate='many_to_one', sort=False)


### MCI-LBD vs HC

In [ ]:
task_hf_compensated_freq_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_lbd_hc[k], task_compensated_dfs_lbd_hc[k])
    for k in task_freq_dfs_lbd_hc.keys() & task_compensated_dfs_lbd_hc.keys()
}

In [ ]:

task_hf_compensated_temp_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_lbd_hc[k], task_compensated_dfs_lbd_hc[k])
    for k in task_freq_dfs_lbd_hc.keys() & task_compensated_dfs_lbd_hc.keys()
}

In [ ]:

task_hf_compensated_comb_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_comb_dfs_lbd_hc[k], task_compensated_dfs_lbd_hc[k])
    for k in task_comb_dfs_lbd_hc.keys() & task_compensated_dfs_lbd_hc.keys()
}

In [ ]:
task_hf_comp_emb_lbl = {
    k: merge_on_subject_agg_right(task_comb_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_comb_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [ ]:
task_hf_freq_emb_lbl = {
    k: merge_on_subject_agg_right(task_freq_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_freq_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [ ]:
task_hf_temp_emb_lbl = {
    k: merge_on_subject_agg_right(task_temp_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_temp_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

### MCI-AD vs HC

In [ ]:
### MCI-AD vs HC
task_hf_compensated_freq_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_hc[k], task_compensated_dfs_ad_hc[k])
    for k in task_freq_dfs_ad_hc.keys() & task_compensated_dfs_ad_hc.keys()
}

task_hf_compensated_temp_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_ad_hc[k], task_compensated_dfs_ad_hc[k])
    for k in task_freq_dfs_ad_hc.keys() & task_compensated_dfs_ad_hc.keys()
}

task_hf_compensated_comb_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_hc[k], task_compensated_dfs_ad_hc[k])
    for k in task_comb_dfs_ad_hc.keys() & task_compensated_dfs_ad_hc.keys()
}
task_hf_comp_emb_lbl_hc_ad= {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_comb_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}
task_hf_freq_emb_lbl = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_freq_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}
task_hf_temp_emb_lbl = {
    k: merge_on_subject_agg_right(task_temp_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_temp_dfs_ad_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

### MCI-LBD vs MCI-AD

In [ ]:
### MCI-LBD vs MCI-LBD
task_hf_compensated_freq_emb_lbl_ad_lbl = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_lbd[k], task_compensated_dfs_ad_lbd[k])
    for k in task_freq_dfs_ad_lbd.keys() & task_compensated_dfs_ad_lbd.keys()
}

task_hf_compensated_temp_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_temp_dfs_ad_lbd[k], task_compensated_dfs_ad_lbd[k])
    for k in task_freq_dfs_ad_lbd.keys() & task_compensated_dfs_ad_lbd.keys()
}

task_hf_compensated_comb_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_lbd[k], task_compensated_dfs_ad_lbd[k])
    for k in task_comb_dfs_ad_lbd.keys() & task_compensated_dfs_ad_lbd.keys()
}
task_hf_comp_emb_lbl_ad_lbd= {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_comb_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}
task_hf_freq_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_freq_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}
task_hf_temp_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_temp_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_temp_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}

### Old dfs from previous analysis

In [ ]:
task_hf_compensated_freq_emb_lbl = {
    k: merge_on_subject_agg_right(task_freq_dfs[k], task_compensated_dfs[k])
    for k in task_freq_dfs.keys() & task_compensated_dfs.keys()
}

In [ ]:
task_hf_compensated_temp_emb_lbl = {
    k: merge_on_subject_agg_right(task_temp_dfs[k], task_compensated_dfs[k])
    for k in task_temp_dfs.keys() & task_compensated_dfs.keys()
}

In [ ]:
task_hf_compensated_comb_emb_lbl = {
    k: merge_on_subject_agg_right(task_comb_dfs[k], task_compensated_dfs[k])
    for k in task_comb_dfs.keys() & task_compensated_dfs.keys()
}

In [ ]:
def x_y_split(df, split_param):
    if split_param is None or split_param == "diagnosis_y":
        #embedding_cols = df[1:-1]
        X = df.iloc[:,1:-2].values
        y = df["diagnosis_y"].values
    elif split_param != None: 
        embedding_cols = [c for c in df.columns if c.startswith(split_param)]
        #X = df[embedding_cols].values
        X = df.iloc[:,1:-2].values
        y = df["diagnosis"].values
    
    return X, y



In [ ]:
def remove_leaking_labels(df, label_string = 'diagnosis_x'):
    df_cleaned = df.copy()
    df_cleaned = df_cleaned.drop(label_string, axis = 1)
    return df_cleaned

In [ ]:
task_hf_compensated_freq_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_hc.items()}
task_hf_compensated_temp_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_hc.items()}
task_hf_compensated_comb_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_hc.items()}

task_hf_compensated_freq_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_lbd_hc.items()}
task_hf_compensated_temp_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_lbd_hc.items()}
task_hf_compensated_comb_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_lbd_hc.items()}


task_hf_compensated_freq_emb_lbl_ad_lbl = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_lbl.items()}
task_hf_compensated_temp_emb_lbl_ad_lbd = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_lbd.items()}
task_hf_compensated_comb_emb_lbl_ad_lbd = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_lbd.items()}

In [ ]:
task_hf_compensated_freq_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_hc.items()}
task_hf_compensated_temp_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_hc.items()}
task_hf_compensated_comb_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_hc.items()}

task_hf_compensated_freq_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_lbd_hc.items()}
task_hf_compensated_temp_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_lbd_hc.items()}
task_hf_compensated_comb_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_lbd_hc.items()}


task_hf_compensated_freq_emb_lbl_ad_lbl = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_lbl.items()}
task_hf_compensated_temp_emb_lbl_ad_lbd = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_lbd.items()}
task_hf_compensated_comb_emb_lbl_ad_lbd = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_lbd.items()}

In [ ]:
def leaking_sanity_test(df, label_string='diagnosis_x'):
    if label_string in df.columns:
        print(f"Leaking label '{label_string}' found in DataFrame columns.")
    else:
        print(f"No leaking label '{label_string}' found in DataFrame columns.")
        

In [ ]:
task_hf_compensated_freq_emb_lbl_ad_hc = {k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_hc.items()}
task_hf_compensated_temp_emb_lbl_ad_hc = {k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_hc.items()}
task_hf_compensated_comb_emb_lbl_ad_hc = {k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_hc.items()}

task_hf_compensated_freq_emb_lbl_lbd_hc = {k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_lbd_hc.items()}
task_hf_compensated_temp_emb_lbl_lbd_hc = {k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_lbd_hc.items()}
task_hf_compensated_comb_emb_lbl_lbd_hc = {k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_lbd_hc.items()}


task_hf_compensated_freq_emb_lbl_ad_lbl = {k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_lbl.items()}
task_hf_compensated_temp_emb_lbl_ad_lbd = {k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_lbd.items()}
task_hf_compensated_comb_emb_lbl_ad_lbd = {k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_lbd.items()}

In [ ]:
def x_y_split_full_df(df, split_param= "diagnosis_y"):
    """
    This function exists solely to divide DataFrame containing both embeddings and handcrafted features
    """
    #embedding_cols = df[1:-1]
    X = df.iloc[:,1:-2].values
    y = df[split_param].values
    
    return X, y

# OPTUNA + XGBoost

In [ ]:

import os, json
import numpy as np
import pandas as pd
import optuna, xgboost as xgb
from typing import Dict, Tuple
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
    matthews_corrcoef
)

# def x_y_split(df): ...  # your splitter (must return X, y)
def _safe_feature_names(X, fallback_dim=None):
    if hasattr(X, "columns"):
        return list(X.columns)
    if hasattr(X, "feature_names_in_"):
        return list(X.feature_names_in_)
    if fallback_dim is None and hasattr(X, "shape"):
        fallback_dim = X.shape[1]
    return [f"f{i}" for i in range(int(fallback_dim or 0))]

def _plot_feature_importance(model, feature_names, out_dir: str):
    """Save multiple feature-importance views from XGBoost."""
    import matplotlib.pyplot as plt
    import numpy as np
    from collections import defaultdict
    os.makedirs(out_dir, exist_ok=True)

    # 1) sklearn API importances (gain-based)
    try:
        importances = getattr(model, "feature_importances_", None)
        if importances is not None and len(importances) == len(feature_names):
            order = np.argsort(importances)[::-1]
            top_idx = order[:50]  # limit to top-50 for readability
            plt.figure()
            plt.barh([feature_names[i] for i in top_idx][::-1], importances[top_idx][::-1])
            plt.title("XGB feature_importances_ (top-50)")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "feature_importances_sklearn.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

    # 2) Booster importances (weight/gain/cover)
    try:
        booster = model.get_booster()
        for typ in ["weight", "gain", "cover", "total_gain", "total_cover"]:
            score_dict = booster.get_score(importance_type=typ)
            if not score_dict:
                continue
            # Map 'f0'.. to friendly names
            vals = []
            names = []
            for k, v in score_dict.items():
                if k.startswith("f"):
                    idx = int(k[1:])
                    if 0 <= idx < len(feature_names):
                        names.append(feature_names[idx])
                    else:
                        names.append(k)
                else:
                    names.append(k)
                vals.append(v)
            order = np.argsort(vals)[::-1]
            top = min(50, len(order))
            plt.figure()
            plt.barh([names[i] for i in order[:top]][::-1], np.array(vals)[order[:top]][::-1])
            plt.title(f"Booster feature importance — {typ} (top-{top})")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"feature_importances_{typ}.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

def _save_shap_summaries(model, X_ref, feature_names, out_dir: str, max_display: int = 30, sample: int = 1000):
    """
    Compute SHAP (TreeExplainer) and save bar + beeswarm plots.
    Skips silently if shap not installed or backend issues arise.
    """
    try:
        import shap
        import numpy as np
        import matplotlib.pyplot as plt
    except Exception:
        return

    try:
        # Downsample for speed
        if hasattr(X_ref, "iloc"):
            X_use = X_ref.sample(min(sample, len(X_ref)), random_state=0)
            X_np = X_use.values
        else:
            X_np = X_ref
            if X_np.shape[0] > sample:
                rng = np.random.default_rng(0)
                idx = rng.choice(X_np.shape[0], size=sample, replace=False)
                X_np = X_np[idx]

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_np)

        # Handle binary classification: shap returns array (n, p)
        if isinstance(shap_values, list) and len(shap_values) == 2:
            # pick positive class
            sv = shap_values[1]
        else:
            sv = shap_values

        # Bar plot (mean |SHAP|)
        try:
            plt.figure()
            shap.summary_plot(sv, X_np, feature_names=feature_names, plot_type="bar", show=False, max_display=max_display)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "shap_summary_bar.png"), dpi=200, bbox_inches="tight")
            plt.close()
        except Exception:
            pass

        # Beeswarm
        try:
            plt.figure()
            shap.summary_plot(sv, X_np, feature_names=feature_names, show=False, max_display=max_display)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "shap_summary_beeswarm.png"), dpi=200, bbox_inches="tight")
            plt.close()
        except Exception:
            pass
    except Exception:
        # Any SHAP error -> skip silently
        return

def _pick_device():
    try:
        _ = xgb.core.get_cuda_compute_capabilities()
        return {"tree_method": "hist", "device": "cuda"}
    except Exception:
        return {"tree_method": "hist", "device": "cpu"}

def _ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def _save_json(d: dict, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(d, f, indent=2, ensure_ascii=False)

def _build_cv(n_splits: int, random_state: int, use_loo: bool = False):
    if use_loo:
        return LeaveOneOut()
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

def make_xgb_objective(X, y, scoring: str = "balanced_accuracy", n_splits: int = 5, random_state: int = 42, use_loo: bool = False):
    if scoring == "balanced_accuracy":
        from sklearn.metrics import balanced_accuracy_score as metric_fn
        needs_proba = False
        eval_metric = "logloss"
    elif scoring == "roc_auc":
        from sklearn.metrics import roc_auc_score as metric_fn
        needs_proba = True
        eval_metric = "auc"
    else:
        raise ValueError("scoring must be 'balanced_accuracy' or 'roc_auc'")

    folds = _build_cv(n_splits=n_splits, random_state=random_state, use_loo=use_loo)
    device_kwargs = _pick_device()

    pos_ratio = float(np.mean(y))
    scale_pos_weight = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0

    def objective(trial: optuna.Trial) -> float:
        params = {
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 200, 1500),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "objective": "binary:logistic",
            "eval_metric": eval_metric,
            "n_jobs": 1,  # keep each trial single-threaded
            "scale_pos_weight": scale_pos_weight,
            **device_kwargs,
        }

        scores = []
        for tr_idx, va_idx in folds.split(X, y):
            if hasattr(X, "iloc"):
                X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
                y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            else:
                X_tr, X_va = X[tr_idx], X[va_idx]
                y_tr, y_va = y[tr_idx], y[va_idx]

            model = xgb.XGBClassifier(**params)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                #early_stopping_rounds=50,
                verbose=False,
            )

            if needs_proba:
                y_score = model.predict_proba(X_va)[:, 1]
                score = metric_fn(y_va, y_score)
            else:
                y_pred = model.predict(X_va)
                score = metric_fn(y_va, y_pred)
            scores.append(score)

            trial.report(np.mean(scores), len(scores))
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(scores))

    return objective

def _evaluate_and_save(name: str, model, X_test, y_test, out_dir: str, feature_names=None, X_ref_for_shap=None):
    """Compute metrics, plot ROC + confusion matrix, and save everything."""
    _ensure_dir(out_dir)

    # Predictions
    y_proba = None
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
    except Exception:
        pass
    y_pred = model.predict(X_test)

    # Metrics
    metrics = {
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "mcc": None,
        "roc_auc": None,
        "sensitivity": None,   # <-- NEW
        "specificity": None,   # <-- NEW
        "classification_report": None,
    }
    # MCC
    try:
        metrics["mcc"] = float(matthews_corrcoef(y_test, y_pred))
    except Exception:
        pass
    # ROC-AUC if both classes present and proba available
    try:
        if y_proba is not None and len(np.unique(y_test)) == 2:
            metrics["roc_auc"] = float(roc_auc_score(y_test, y_proba))
        else:
            metrics["roc_auc"] = None
    except Exception:
        metrics["roc_auc"] = None
    # --- Sensitivity & Specificity (labels fixed to [0,1]) ---
    try:
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        sens_den = tp + fn
        spec_den = tn + fp
        metrics["sensitivity"] = float(tp / sens_den) if sens_den > 0 else None
        metrics["specificity"] = float(tn / spec_den) if spec_den > 0 else None
    except Exception:
        pass


    # Classification report (as dict)
    try:
        metrics["classification_report"] = classification_report(y_test, y_pred, output_dict=True)
    except Exception:
        metrics["classification_report"] = None

    # Confusion matrix plot
    try:
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(cm)
        import matplotlib.pyplot as plt
        plt.figure()
        disp.plot(values_format="d")
        plt.title(f"Confusion Matrix — {name}")
        plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=200, bbox_inches="tight")
        plt.close()
    except Exception:
        pass

    # ROC curve plot
    try:
        if y_proba is not None and len(np.unique(y_test)) == 2:
            import matplotlib.pyplot as plt
            plt.figure()
            RocCurveDisplay.from_predictions(y_test, y_proba)
            plt.title(f"ROC Curve — {name}")
            plt.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

    # Feature names
    if feature_names is None:
        feature_names = _safe_feature_names(X_test)

    # Feature importance plots
    try:
        _plot_feature_importance(model, feature_names, out_dir)
    except Exception:
        pass

    # SHAP plots (optional)
    try:
        if X_ref_for_shap is None:
            X_ref_for_shap = X_test
        _save_shap_summaries(model, X_ref_for_shap, feature_names, out_dir)
    except Exception:
        pass

    # Save metrics JSON
    _save_json(metrics, os.path.join(out_dir, "metrics.json"))
    return metrics

def _evaluate_cv_and_save(
    name: str,
    X,
    y,
    params: dict,
    scoring: str,
    n_splits: int,
    random_state: int,
    out_dir: str,
    use_loo: bool = False,
):
    """Evaluate metrics using CV on the full dataset and save plots/metrics."""
    _ensure_dir(out_dir)

    folds = _build_cv(n_splits=n_splits, random_state=random_state, use_loo=use_loo)

    y_true_all = []
    y_pred_all = []
    y_proba_all = []
    has_proba = True
    fold_metrics = []

    for tr_idx, va_idx in folds.split(X, y):
        if hasattr(X, "iloc"):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        else:
            X_tr, X_va = X[tr_idx], X[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            #early_stopping_rounds=50,
            verbose=False,
        )

        y_pred = model.predict(X_va)
        y_proba = None
        if has_proba:
            try:
                y_proba = model.predict_proba(X_va)[:, 1]
            except Exception:
                has_proba = False
                y_proba = None

        # Fold-level metrics
        fold_result = {
            "balanced_accuracy": float(balanced_accuracy_score(y_va, y_pred)),
            "mcc": None,
            "roc_auc": None,
            "sensitivity": None,
            "specificity": None,
        }
        try:
            fold_result["mcc"] = float(matthews_corrcoef(y_va, y_pred))
        except Exception:
            pass
        try:
            if y_proba is not None and len(np.unique(y_va)) == 2:
                fold_result["roc_auc"] = float(roc_auc_score(y_va, y_proba))
        except Exception:
            pass
        try:
            cm = confusion_matrix(y_va, y_pred, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()
            sens_den = tp + fn
            spec_den = tn + fp
            fold_result["sensitivity"] = float(tp / sens_den) if sens_den > 0 else None
            fold_result["specificity"] = float(tn / spec_den) if spec_den > 0 else None
        except Exception:
            pass
        fold_metrics.append(fold_result)

        y_true_all.append(y_va)
        y_pred_all.append(y_pred)
        if has_proba:
            y_proba_all.append(y_proba)

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)
    y_proba_all = np.concatenate(y_proba_all) if has_proba and len(y_proba_all) else None

    # Pooled metrics
    metrics = {
        "balanced_accuracy": float(balanced_accuracy_score(y_true_all, y_pred_all)),
        "mcc": None,
        "roc_auc": None,
        "sensitivity": None,
        "specificity": None,
        "classification_report": None,
        "cv_stats": {},
    }
    try:
        metrics["mcc"] = float(matthews_corrcoef(y_true_all, y_pred_all))
    except Exception:
        pass
    try:
        if y_proba_all is not None and len(np.unique(y_true_all)) == 2:
            metrics["roc_auc"] = float(roc_auc_score(y_true_all, y_proba_all))
    except Exception:
        pass
    try:
        cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        sens_den = tp + fn
        spec_den = tn + fp
        metrics["sensitivity"] = float(tp / sens_den) if sens_den > 0 else None
        metrics["specificity"] = float(tn / spec_den) if spec_den > 0 else None
    except Exception:
        pass
    try:
        metrics["classification_report"] = classification_report(y_true_all, y_pred_all, output_dict=True)
    except Exception:
        metrics["classification_report"] = None

    # Fold summary stats
    for key in ["balanced_accuracy", "mcc", "roc_auc", "sensitivity", "specificity"]:
        vals = [m.get(key) for m in fold_metrics if m.get(key) is not None]
        metrics["cv_stats"][f"{key}_mean"] = float(np.mean(vals)) if vals else None
        metrics["cv_stats"][f"{key}_std"] = float(np.std(vals)) if vals else None

    # Confusion matrix plot
    try:
        cm = confusion_matrix(y_true_all, y_pred_all)
        disp = ConfusionMatrixDisplay(cm)
        import matplotlib.pyplot as plt
        plt.figure()
        disp.plot(values_format="d")
        plt.title(f"Confusion Matrix ? {name}")
        plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=200, bbox_inches="tight")
        plt.close()
    except Exception:
        pass

    # ROC curve plot
    try:
        if y_proba_all is not None and len(np.unique(y_true_all)) == 2:
            import matplotlib.pyplot as plt
            plt.figure()
            RocCurveDisplay.from_predictions(y_true_all, y_proba_all)
            plt.title(f"ROC Curve ? {name}")
            plt.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

    _save_json(metrics, os.path.join(out_dir, "metrics.json"))
    return metrics

def optimize_many(
    dfs: Dict[str, pd.DataFrame],
    x_y_split_fn,
    scoring: str = "balanced_accuracy",
    n_trials: int = 500,
    n_splits: int = 10,
    random_state: int = 42,
    n_jobs_trials: int = -1,
    output_dir: str = "xgb_results",
    test_size: float = 0.20,
    split_param: str = "emb_",
    use_loo: bool = False,
) -> Tuple[pd.DataFrame, Dict[str, xgb.XGBClassifier], Dict[str, optuna.Study]]:
    """
    Runs Optuna per dataset on the full dataset and validates via CV,
    saving plots + metrics in output_dir/<dataset_name>/
    """
    results = []
    best_models: Dict[str, xgb.XGBClassifier] = {}
    studies: Dict[str, optuna.Study] = {}

    _ensure_dir(output_dir)

    for name, df in dfs.items():
        X, y = x_y_split_fn(df, split_param=split_param)

        # To numpy for fast indexing, but keep DataFrame support
        X_arr = X.values if hasattr(X, "values") else X
        y_arr = y.values if hasattr(y, "values") else y

        feature_names = _safe_feature_names(X)
        # Keep names before converting to numpy

        # Build objective on full dataset
        objective = make_xgb_objective(
            X_arr, y_arr,
            scoring=scoring,
            n_splits=n_splits,
            random_state=random_state,
            use_loo=use_loo,
        )

        study = optuna.create_study(
            study_name=f"{name}_{scoring}",
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=random_state),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=10),
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True, n_jobs=n_jobs_trials)
        studies[name] = study

        # Train final model on the full dataset using best params
        device_kwargs = _pick_device()
        pos_ratio = float(np.mean(y_arr))
        spw = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0

        best_params = {
            **study.best_params,
            "objective": "binary:logistic",
            "eval_metric": "auc" if scoring == "roc_auc" else "logloss",
            "n_jobs": 0,
            "scale_pos_weight": spw,
            **device_kwargs,
        }
        final_model = xgb.XGBClassifier(**best_params)
        final_model.fit(X_arr, y_arr, verbose=False)

        # Save per-dataset artifacts
        ds_out = os.path.join(output_dir, name)
        _ensure_dir(ds_out)

        # CV evaluation on full dataset
        metrics = _evaluate_cv_and_save(
            name,
            X_arr,
            y_arr,
            best_params,
            scoring=scoring,
            n_splits=n_splits,
            random_state=random_state,
            out_dir=ds_out,
            use_loo=use_loo,
        )

        # Save best params + study summary
        _save_json({"best_params": study.best_params, "best_value": study.best_value}, os.path.join(ds_out, "best.json"))

        # Feature importance plots
        try:
            _plot_feature_importance(final_model, feature_names, ds_out)
        except Exception:
            pass

        # SHAP plots (optional)
        try:
            _save_shap_summaries(final_model, X_arr, feature_names, ds_out)
        except Exception:
            pass

        # Optional: save model
        try:
            import joblib
            joblib.dump(final_model, os.path.join(ds_out, "model.joblib"))
        except Exception:
            pass

        best_models[name] = final_model
        results.append({
            "dataset": name,
            "cv_best_value": study.best_value,
            "cv_balanced_accuracy": metrics.get("balanced_accuracy"),
            "cv_roc_auc": metrics.get("roc_auc"),
            "n_trials": len(study.trials),
        })

    summary_df = pd.DataFrame(results).sort_values(by="cv_balanced_accuracy", ascending=False).reset_index(drop=True)
    # Save a global summary too
    summary_df.to_csv(os.path.join(output_dir, "summary.csv"), index=False)
    return summary_df, best_models, studies


## MCI-LBD vs HC -> OPTUNA + XGB 

In [ ]:
datasets = task_freq_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_results_freq",
    split_param="emb_",
)
print(summary)


In [ ]:
# Temporal embedings
datasets = task_temp_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="roc_auc",      
    n_trials=200,
    n_splits=5,
    output_dir="xgb_results_temporal",
    split_param="emb_",
)
print(summary)


In [ ]:
# Combined embedings
datasets = task_comb_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="roc_auc",     # or "roc_auc"
    n_trials=200,
    n_splits=5,
    output_dir="xgb_results_combined",
    split_param="emb_",
)
print(summary)


In [ ]:

datasets = task_hf_dfs_clean_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_results_handcrafted",
    split_param="w.cz.fnusa",
)
print(summary)

# Task - wise OPTUNA + XGBoost (atomatic saving) - embedding and handcrafted features combined

In [ ]:

datasets = task_hf_emb_lbl

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_results_handcrafted_embedded",
    split_param= None,
)
print(summary)

# Task - wise OPTUNA + XGBoost (atomatic saving) - embedding and handcrafted features combined

In [ ]:
# Task - wise OPTUNA + XGBoost (atomatic saving) - embedding and handcrafted features combined

datasets = task_compensated_dfs_clean_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_results_handcrafted_compensated",
    split_param= "diagnosis",
)
print(summary)

In [ ]:

datasets = task_hf_compensated_comb_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_results_handcrafted_combined_emb_compensated",
    split_param= "diagnosis_y",
)
print(summary)

In [ ]:

datasets = task_hf_compensated_temp_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_results_handcrafted_temporal_emb_compensated",
    split_param= "diagnosis_y",
)
print(summary)

In [ ]:
datasets = task_hf_compensated_freq_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_results_handcrafted_frerq_emb_compensated",
    split_param= "diagnosis_y",
)
print(summary)

# MCI_LBD vs HC -> OPTUNA + XGB

In [ ]:
# Task - wise OPTUNA + XGBoost (atomatic saving) - embedding and handcrafted features combined

datasets = task_compensated_dfs_clean_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_compensated",
    split_param= "diagnosis",
)
print(summary)


# From this one

In [ ]:

datasets = task_hf_compensated_comb_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_combined_emb_compensated",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:
task_hf_compensated_comb_emb_lbl_lbd_hc.get('1_1')

In [ ]:

datasets = task_hf_compensated_temp_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_temporal_emb_compensated",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:
task_hf_compensated_temp_emb_lbl_lbd_hc.get('18_1').head(15)

In [ ]:

datasets = task_hf_compensated_freq_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_frerq_emb_compensated",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

In [ ]:
datasets = task_freq_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_freq_ad_hc",
    split_param="emb_",
)
print(summary)


In [ ]:

# Temporal embedings
datasets = task_temp_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="roc_auc",      
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_temporal",
    split_param="emb_",
)
print(summary)


In [ ]:

# Combined embedings
datasets = task_comb_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="roc_auc",     # or "roc_auc"
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_combined",
    split_param="emb_",
)
print(summary)



In [ ]:

datasets = task_hf_dfs_clean_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_handcrafted",
    split_param="w.cz.fnusa",
)
print(summary)

## MCI-AD vs HC -> OPTUNA + XGB

In [ ]:
# Task - wise OPTUNA + XGBoost (atomatic saving) - embedding and handcrafted features combined

datasets = task_compensated_dfs_clean_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_handcrafted_compensated",
    split_param= "diagnosis",
)
print(summary)


In [ ]:

datasets = task_hf_compensated_comb_emb_lbl_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_handcrafted_combined_emb_compensated",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:
task_hf_compensated_temp_emb_lbl_ad_hc.get('9_1')

In [ ]:

datasets = task_hf_compensated_temp_emb_lbl_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_handcrafted_temporal_emb_compensated",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:

datasets = task_hf_compensated_freq_emb_lbl_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_handcrafted_frerq_emb_compensated",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

In [ ]:
datasets = task_freq_dfs_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_freq_ad_hc",
    split_param="emb_",
)
print(summary)


In [ ]:

# Temporal embedings
datasets = task_temp_dfs_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="roc_auc",      
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_temporal",
    split_param="emb_",
)
print(summary)


In [ ]:

# Combined embedings
datasets = task_comb_dfs_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="roc_auc",     # or "roc_auc"
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_combined",
    split_param="emb_",
)
print(summary)



In [ ]:

datasets = task_hf_dfs_clean_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_handcrafted",
    split_param="w.cz.fnusa",
)
print(summary)

## MCI-LBD vs MCI-AD -> OTPUNA + XBG

In [ ]:
# Task - wise OPTUNA + XGBoost (atomatic saving) - embedding and handcrafted features combined

datasets = task_compensated_dfs_clean_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_compensated_ad_lbd",
    split_param= "diagnosis",
)
print(summary)


In [ ]:

datasets = task_hf_compensated_comb_emb_lbl_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_combined_emb_compensated_ad_lbd",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:

datasets = task_hf_compensated_temp_emb_lbl_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_temporal_emb_compensated_ad_lbd",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:

datasets = task_hf_compensated_freq_emb_lbl_ad_lbl

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_frerq_emb_compensated_ad_lbd",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

In [ ]:
datasets = task_freq_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_freq_ad_lbd",
    split_param="emb_",
)
print(summary)


In [ ]:

# Temporal embedings
datasets = task_temp_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="roc_auc",      
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_temporal_ad_lbd",
    split_param="emb_",
)
print(summary)


In [ ]:

# Combined embedings
datasets = task_comb_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="roc_auc",     # or "roc_auc"
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_combined_ad_lbd",
    split_param="emb_",
)
print(summary)



In [ ]:

datasets = task_hf_dfs_clean_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_ad_lbd",
    split_param="w.cz.fnusa",
)
print(summary)